In [2]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║              98_replay_training_output.ipynb                               ║
# ║  Reconstruct the per-epoch training table from saved .npy history files    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── CELL 1: Imports and config ────────────────────────────────────────────────
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("../results")

In [3]:
# ── CELL 2: Core replay function ──────────────────────────────────────────────
def replay_run(npy_path: Path):
    """
    Load a history .npy file and print the epoch table exactly as it
    appeared during training, plus the best-epoch summary line.

    Expected history dict keys (adjust if yours differ):
      train_loss, val_loss, val_f1, val_f1_nobg, val_sens, val_spec, lr
    """
    h        = np.load(npy_path, allow_pickle=True).item()
    run_name = npy_path.stem.replace("_history", "")

    # ── Determine available keys ──────────────────────────────────────────────
    train_loss  = h.get("train_loss",  [])
    val_loss    = h.get("val_loss",    [])
    val_f1      = h.get("val_f1",      [])
    val_f1_nobg = h.get("val_f1_nobg", h.get("val_f1_no_bg", []))
    val_dice_nobg = h.get("val_dice_nobg", val_f1_nobg)  # often identical
    val_sens    = h.get("val_sens",    h.get("val_sensitivity", []))
    val_spec    = h.get("val_spec",    h.get("val_specificity", []))
    lr_hist     = h.get("lr",          [])

    n_epochs = len(train_loss)
    if n_epochs == 0:
        print(f"  ⚠️  {run_name}: empty history")
        return

    # ── Pad missing sequences with None ──────────────────────────────────────
    def pad(seq, n): return list(seq) + [None] * (n - len(seq))

    val_loss    = pad(val_loss,    n_epochs)
    val_f1      = pad(val_f1,      n_epochs)
    val_f1_nobg = pad(val_f1_nobg, n_epochs)
    val_dice_nobg = pad(val_dice_nobg, n_epochs)
    val_sens    = pad(val_sens,    n_epochs)
    val_spec    = pad(val_spec,    n_epochs)
    lr_hist     = pad(lr_hist,     n_epochs)

    # ── Best epoch (by val_f1_nobg) ───────────────────────────────────────────
    valid_f1    = [(i, v) for i, v in enumerate(val_f1_nobg) if v is not None]
    best_idx    = max(valid_f1, key=lambda x: x[1])[0] if valid_f1 else 0
    best_f1     = val_f1_nobg[best_idx]
    best_dice   = val_dice_nobg[best_idx]
    best_f1all  = val_f1[best_idx]
    best_sens   = val_sens[best_idx]
    best_spec   = val_spec[best_idx]

    # ── Print header ─────────────────────────────────────────────────────────
    print(f"\n{'═'*60}")
    print(f"  Run: {run_name}")
    print(f"{'═'*60}")
    print()
    print(f"  {'Epoch':>7}   {'Train Loss':>10}   {'Val Loss':>9}   "
          f"{'Val F1':>8}   {'F1 -BG':>7}   {'Dice -BG':>8}   "
          f"{'Sens':>7}   {'Spec':>7}   {'LR':>10}")
    print(f"  {'─'*107}")

    for i in range(n_epochs):
        ep       = i + 1
        tl       = f"{train_loss[i]:.4f}"   if train_loss[i]  is not None else "    —   "
        vl       = f"{val_loss[i]:.4f}"     if val_loss[i]    is not None else "    —   "
        vf1      = f"{val_f1[i]:.4f}"       if val_f1[i]      is not None else "    —   "
        vf1nb    = f"{val_f1_nobg[i]:.4f}"  if val_f1_nobg[i] is not None else "    —   "
        vdnb     = f"{val_dice_nobg[i]:.4f}" if val_dice_nobg[i] is not None else "    —   "
        vs       = f"{val_sens[i]:.4f}"     if val_sens[i]    is not None else "    —   "
        vsp      = f"{val_spec[i]:.4f}"     if val_spec[i]    is not None else "    —   "
        lr       = f"{lr_hist[i]:.2e}"      if lr_hist[i]     is not None else "    —   "
        star     = " ★" if i == best_idx else "  "

        print(f"  {ep:>7}       {tl:>10}   {vl:>9}   "
              f"{vf1:>8}   {vf1nb:>7}   {vdnb:>8}   "
              f"{vs:>7}   {vsp:>7}   {lr:>10}{star}")

    print(f"\n  ✅ Best val macro-F1 (no BG) : {best_f1:.4f}  "
          f"Dice (no BG): {best_dice:.4f}  "
          f"F1 (all): {best_f1all:.4f}  "
          f"Sens: {best_sens:.4f}  "
          f"Spec: {best_spec:.4f}  "
          f"@ epoch {best_idx + 1}")

In [4]:
# ── CELL 3: Inspect a single run ──────────────────────────────────────────────
# Useful to check history keys before replaying everything
def inspect_keys(npy_path: Path):
    h = np.load(npy_path, allow_pickle=True).item()
    print(f"Keys in {npy_path.name}:")
    for k, v in h.items():
        length = len(v) if hasattr(v, '__len__') else 'scalar'
        sample = v[0] if hasattr(v, '__len__') and len(v) > 0 else v
        print(f"  {k:<25} len={length}   sample={sample}")

# Run this on one file first to confirm your key names
first_npy = next(RESULTS_DIR.glob("*_history.npy"), None)
if first_npy:
    inspect_keys(first_npy)

Keys in 1dnn_ufl_bal_fold2_vpfabelo_history.npy:
  train_loss                len=27   sample=0.46215730261802673
  val_loss                  len=27   sample=0.327349742325482
  val_f1                    len=27   sample=0.6422506798660882
  val_f1_no_bg              len=27   sample=0.5869756166157337
  val_sens                  len=27   sample=0.7000691448147006
  val_spec                  len=27   sample=0.9255820722784729
  val_dice                  len=27   sample=0.6422506798561839
  val_dice_no_bg            len=27   sample=0.5869756166040244
  best_epoch                len=scalar   sample=12
  best_f1                   len=scalar   sample=0.836914863991712
  best_f1_all               len=scalar   sample=0.8675840133309252
  best_sens                 len=scalar   sample=0.8959768448996572
  best_spec                 len=scalar   sample=0.9774965997447015
  best_dice                 len=scalar   sample=0.8675840133109176
  best_dice_no_bg           len=scalar   sample=0.836914863966

In [5]:
def replay_all(results_dir: Path, filter_prefix: str = ""):
    """
    Replay all history files in results_dir.
    Optionally filter by prefix, e.g. "2dcnn" for 2D runs only.
    """
    paths = sorted(results_dir.glob("*_history.npy"))
    if filter_prefix:
        paths = [p for p in paths if p.stem.startswith(filter_prefix)]

    print(f"Replaying {len(paths)} runs"
          + (f" matching '{filter_prefix}*'" if filter_prefix else ""))

    for path in paths:
        replay_run(path)

    # ── Summary table at the end ──────────────────────────────────────────────
    print(f"\n\n{'═'*60}")
    print(f"  SUMMARY")
    print(f"{'═'*60}")
    print(f"  {'Run':<50}   {'Best F1-noBG':>12}   {'Best epoch':>10}")
    print(f"  {'─'*77}")

    for path in paths:
        h           = np.load(path, allow_pickle=True).item()
        run_name    = path.stem.replace("_history", "")
        val_f1_nobg = h.get("val_f1_nobg", h.get("val_f1_no_bg", []))
        if val_f1_nobg:
            best_ep  = int(np.argmax(val_f1_nobg)) + 1
            best_val = float(np.max(val_f1_nobg))
            print(f"  {run_name:<50}   {best_val:>12.4f}   {best_ep:>10}")
        else:
            print(f"  {run_name:<50}   {'no data':>12}")

    print(f"{'═'*60}")


# Reply a specific run
# replay_run(RESULTS_DIR / "1dnnfabelo_ce_bal_fold1_vpfabelo_history.npy")

# # Replay all 2D runs
# replay_all(RESULTS_DIR, filter_prefix="2dcnn")

# Or replay everything:
# replay_all(RESULTS_DIR)

In [7]:
replay_all(RESULTS_DIR, filter_prefix="spectralformer_caf_ce_bal")

Replaying 5 runs matching 'spectralformer_caf_ce_bal*'

════════════════════════════════════════════════════════════
  Run: spectralformer_caf_ce_bal_fold1_vpfabelo
════════════════════════════════════════════════════════════

    Epoch   Train Loss    Val Loss     Val F1    F1 -BG   Dice -BG      Sens      Spec           LR
  ───────────────────────────────────────────────────────────────────────────────────────────────────────────
        1           1.1732      1.1901     0.3221    0.1869     0.1869    0.3855    0.8789         —     
        2           0.9194      1.5051     0.3331    0.2101     0.2101    0.4986    0.8669         —     
        3           0.6779      0.6357     0.6169    0.5248     0.5248    0.6250    0.9238         —     
        4           0.5994      0.9434     0.5210    0.4088     0.4088    0.6054    0.9213         —     
        5           0.4689      0.7557     0.5873    0.4802     0.4802    0.7008    0.9321         —     
        6           0.4514      0